# PE：位置嵌入

## 绝对位置嵌入
- 使用另一个维度相同的向量，其中每个向量唯一表示句子中的一个位置，输入是通过将词嵌入与其对应的位置嵌入相加得到；
- 正余弦编码；
- 局限：
    - 序列长度有限，无法超出限制；
    - 位置嵌入的独立性：每个位置嵌入都与其他位置嵌入相互独立，即模型认为位置1和位置2之间的差异与位置2和位置500之间的差异相同，然而前者应该更密切，缺失这种相对位置信息会阻碍细微差别；
- 正余弦编码虽然天然蕴含相对信息，但是，（1）由于加法注入，计算 $QK^T$ 时，词内容与位置的交叉乘积会产生干扰，产生大量噪音；（2）$PE_i W_Q W_K^T PE_j^T$ 项中，中间夹杂的 $W_Q W_K^T$ 项带来的线性映射破坏了 $PE_i PE_j^T$ 的几何关系，导致深层网络很难恢复出原本的相对位置。

## 相位位置嵌入
- 侧重于词元对之间的距离，通过改变注意力机制来整合相对信息；
- **位置偏移偏差**：使用一个偏差值（浮点数）来表示每个可能的位置偏移量；
- **在自注意力层中整合**：相对位置偏差矩阵添加到自注意力层中Q和K矩阵的乘积中，确保相对距离相同的词元始终由相同的偏差表示，而与其在序列中的位置无关；
- **可拓展性**：可以拓展到任意长的序列；
- 局限：
    - 性能问题：慢，尤其是处理较长序列时，因为自注意力层中额外的计算步骤（即位置矩阵添加到Q、K矩阵中）；
    - 键值缓存使用复杂性：每个新增的标记都会改变其他所有标记的嵌入，使得transformer模型中有效使用键值缓存变得复杂

## RoPE：Rotary Positional Embedding，旋转位置嵌入
- 注意力计算时$qk^T$之间的点积计算并没有考虑token的位置信息，通过旋转位置编码，在attention中添加位置信息
    - 通过sin和cos函数构成的旋转矩阵，对向量进行旋转。定义旋转矩阵$R(\theta)$：
    $$
    R(\theta)=\begin{bmatrix}
    cos(\theta) & -sin(\theta) \\
    sin(\theta) & cos(\theta)
    \end{bmatrix}
    $$
    - 任意向量x，可以用$xR(\theta)$表示对该向量进行逆时针旋转θ角度
    - 结合性：$R(\theta_1)R(\theta_2)=R(\theta_1+\theta_2)$
    - $R(\theta)^T=R(-\theta)$
    - 正交矩阵：$R(\theta)R(\theta)^T=R(\theta)^TR(\theta)=E$,故不会改变向量的模长，因此不会改变原模型的稳定
    - 通过旋转矩阵对两个向量进行编码来添加位置信息，对每个向量根据它们的位置索引（分别为m和n）进行旋转。如果对q应用旋转矩阵R(m)，对k应用旋转矩阵R(n)，然后再进行点积计算：
    $qR(m)\cdot (kR(n))^T=qR(m)R(n)^Tk^T=qR(m-n)k^T$
      是天然的相对位置编码，而且因为位置信息是一个连续函数，所以可外推
    - 拓展到高维：上面说的是2维旋转矩阵，拓展到高维，将维度两两一组进行旋转（任意两个特征一组都可以），每一组在它们两个特征组成的子空间内进行旋转
    $$
    R(\theta,m)=\begin{bmatrix}
    cos(m\theta_0) & -sin(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
    sin(m\theta_0) & cos(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
    0 & 0 & cos(m\theta_1) & -sin(m\theta_1) & \cdots & 0 & 0 \\
    0 & 0 & sin(m\theta_1) & cos(m\theta_1) & \cdots & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \cdots & cos(m\theta_{/frac{d}{2}-1}) & -sin(m\theta_{/frac{d}{2}-1}) \\
    0 & 0 & 0 & 0 & \cdots & sin(m\theta_{/frac{d}{2}-1}) & cos(m\theta_{/frac{d}{2}-1}) \\
    \end{bmatrix}
    $$
    \*其中，m代表该token在序列中的位置，$\theta_i=\frac{1}{10000^{\frac{2i}{d}}}$
    - 由于$R(\theta,m)$的稀疏性，所以一般不通过矩阵乘法进行计算，而是采用简单的**向量运算**实现：
    $$
    R(\theta,m)x=
    \begin{bmatrix}
    cos{m\theta_0} \\ cos{m\theta_0} \\
    cos{m\theta_1} \\ cos{m\theta_1} \\
    \vdots \\
    cos{m\theta_{\frac{d}{2}-1}} \\ cos{m\theta_{\frac{d}{2}-1}} \\
    \end{bmatrix}
    \otimes
    \begin{bmatrix}
    x_0 \\ x_1 \\ x_2 \\ x_3 \\ \vdots \\ x_{d-2} \\ x_{d-1} \\
    \end{bmatrix}
    +
    \begin{bmatrix}
    sin{m\theta_0} \\ sin{m\theta_0} \\
    sin{m\theta_1} \\ sin{m\theta_1} \\
    \vdots \\
    sin{m\theta_{\frac{d}{2}-1}} \\ sin{m\theta_{\frac{d}{2}-1}} \\
    \end{bmatrix}
    \otimes
    \begin{bmatrix}
    -x_1 \\ x_0 \\ -x_3 \\ x_2 \\ \vdots \\ -x_{d-1} \\ x_{d-2} \\
    \end{bmatrix}
    $$
    \* 其中，x是token的特征向量，$x_i$表示特征向量第i个位置的值